# **BOOTCAMP SPACE TECH — PYTHON PARA GEOINFORMÁTICA**

# Módulo 5: NumPy para Dados Espaciais

NumPy é a biblioteca fundamental para computação científica em Python. Em geoinformática e sensoriamento remoto, trabalhamos constantemente com arrays multidimensionais — bandas espectrais de satélites, matrizes de elevação, séries temporais de índices de vegetação. O NumPy é a base de todo esse processamento.

## Objetivos de Aprendizagem

* Compreender o que é NumPy e por que é essencial para dados espaciais
* Criar e manipular arrays NumPy
* Realizar operações vetorizadas (band math)
* Indexar e fatiar arrays multidimensionais
* Aplicar funções estatísticas em dados de satélite
* Trabalhar com coordenadas espaciais como arrays

## 1. Por que NumPy para Geoinformática?

Imagens de satélite são essencialmente arrays multidimensionais:
- Uma banda espectral = array 2D (linhas x colunas)
- Uma imagem multiespectral = array 3D (bandas x linhas x colunas)
- Uma série temporal de imagens = array 4D

Operações como NDVI, classificação de cobertura do solo e detecção de mudanças são todas operações de array.

In [ ]:
import numpy as np

print(f"NumPy versão {np.__version__}")

## 2. Criando Arrays — Simulando Dados de Satélite

Vamos simular dados reais de sensoriamento remoto. Uma imagem Landsat, por exemplo, possui pixels com valores de reflectância entre 0 e 1 (ou 0-10000 em valores DN).

In [ ]:
np.random.seed(42)

# Simulando uma banda espectral (ex: banda do vermelho) de 500x500 pixels
banda_vermelho = np.random.random((500, 500)) * 0.5 + 0.1
banda_nir = np.random.random((500, 500)) * 0.6 + 0.2

print(f"Formato da banda vermelho: {banda_vermelho.shape}")
print(f"Tipo de dados: {banda_vermelho.dtype}")
print(f"Valor mínimo: {banda_vermelho.min():.4f}, máximo: {banda_vermelho.max():.4f}")

### Criando arrays de diferentes formas

In [ ]:
# Array de zeros — útil para inicializar máscaras de nuvens
mascara_nuvens = np.zeros((100, 100))
print("Máscara de nuvens (zeros):", mascara_nuvens.shape)

# Array de uns — útil para áreas de interesse
area_estudo = np.ones((50, 50))
print("Área de estudo (uns):", area_estudo.shape)

# np.arange — sequências para coordenadas
latitudes = np.arange(-33.0, -30.0, 0.01)
print(f"Coordenadas de latitude: {len(latitudes)} pontos, de {latitudes[0]:.2f} a {latitudes[-1]:.2f}")

# np.linspace — número fixo de pontos igualmente espaçados
longitudes = np.linspace(-54.0, -50.0, 400)
print(f"Coordenadas de longitude: {len(longitudes)} pontos")

# np.meshgrid — grade de coordenadas (essencial para mapas!)
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)
print(f"Grade de coordenadas: lon {lon_grid.shape}, lat {lat_grid.shape}")

## 3. Operações Vetorizadas — Band Math

O poder do NumPy está nas operações vetorizadas — aplicar cálculos em arrays inteiros sem loops. Isto é fundamental para processar imagens de satélite eficientemente.

In [ ]:
# Operações aritméticas em arrays
# Convertendo reflectância para valores DN (Digital Number)
dn_vermelho = banda_vermelho * 10000
print(f"DN vermelho — min: {dn_vermelho.min():.0f}, max: {dn_vermelho.max():.0f}")

### Índice de Vegetação (NDVI)

O NDVI (Normalized Difference Vegetation Index) é um dos índices mais usados em sensoriamento remoto:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

Onde NIR é a banda do infravermelho próximo e Red é a banda do vermelho. O NDVI varia de -1 a 1, onde valores maiores indicam vegetação mais densa e saudável.

In [ ]:
# Calculando NDVI
ndvi = (banda_nir - banda_vermelho) / (banda_nir + banda_vermelho + 1e-10)

print(f"NDVI — min: {ndvi.min():.4f}, max: {ndvi.max():.4f}")
print(f"NDVI — média: {ndvi.mean():.4f}")

# Quantos pixels tem vegetação densa? (NDVI > 0.6)
vegetacao_densa = np.sum(ndvi > 0.6)
total_pixels = ndvi.size
print(f"Pixels com vegetação densa: {vegetacao_densa} de {total_pixels}")
print(f"Percentual: {vegetacao_densa/total_pixels*100:.1f}%")

# Classificação simples de cobertura
agua = np.sum(ndvi < 0)
solo_exposto = np.sum((ndvi >= 0) & (ndvi < 0.2))
vegetacao_rala = np.sum((ndvi >= 0.2) & (ndvi < 0.5))
vegetacao_densa = np.sum(ndvi >= 0.5)
print(f"\nClassificação de cobertura:")
print(f"  Água: {agua} pixels ({agua/total_pixels*100:.1f}%)")
print(f"  Solo exposto: {solo_exposto} pixels ({solo_exposto/total_pixels*100:.1f}%)")
print(f"  Vegetação rala: {vegetacao_rala} pixels ({vegetacao_rala/total_pixels*100:.1f}%)")
print(f"  Vegetação densa: {vegetacao_densa} pixels ({vegetacao_densa/total_pixels*100:.1f}%)")

### Operações com múltiplas bandas — Imagem multiespectral

In [ ]:
# Simulando uma imagem Landsat 8 com 7 bandas
nomes_bandas = ['Costeira', 'Azul', 'Verde', 'Vermelho', 'NIR', 'SWIR1', 'SWIR2']
imagem_landsat = np.random.random((7, 500, 500)) * 0.5 + 0.1
print(f"Imagem Landsat simulada: {imagem_landsat.shape}")

# Média espectral de cada banda
print("\nReflectância média por banda:")
for i, nome in enumerate(nomes_bandas):
    print(f"  {nome}: {imagem_landsat[i].mean():.4f}")

# Estatísticas da banda NIR (índice 4)
b4 = imagem_landsat[4]
print(f"\nEstatísticas da banda NIR:")
print(f"  Mín: {b4.min():.4f}, Máx: {b4.max():.4f}")
print(f"  Média: {b4.mean():.4f}, Desvio padrão: {b4.std():.4f}")
print(f"  Mediana: {np.median(b4):.4f}")
print(f"  Percentil 90: {np.percentile(b4, 90):.4f}")

## 4. Indexação e Fatiamento — Extraindo Regiões de Interesse

Em geoinformática, frequentemente precisamos extrair sub-regiões de imagens (ROI — Region of Interest) para análise localizada.

In [ ]:
# Extraindo uma ROI (janela de 100x100 pixels do canto superior esquerdo)
roi = banda_vermelho[:100, :100]
print(f"ROI extraída: {roi.shape}")

# Extraindo o centro da imagem (100x100 pixels centrais)
h, w = banda_vermelho.shape
centro_y, centro_x = h // 2, w // 2
centro = banda_vermelho[centro_y-50:centro_y+50, centro_x-50:centro_x+50]
print(f"Centro da imagem: {centro.shape}")

# Comparando estatísticas da ROI central com a imagem completa
print(f"\nMédia da ROI central: {centro.mean():.4f}")
print(f"Média da imagem completa: {banda_vermelho.mean():.4f}")

# Indexação booleana — extrair pixels que atendem a uma condição
pixels_brilhantes = banda_vermelho[banda_vermelho > 0.5]
print(f"\nPixels com reflectância > 0.5: {len(pixels_brilhantes)}")

## 5. Manipulação de Arrays — Remodelagem e Empilhamento

Dados espaciais frequentemente precisam ser reorganizados para diferentes análises.

In [ ]:
# reshape — reorganizar um array 2D em 1D (útil para algoritmos de ML)
pixels_1d = banda_vermelho.reshape(-1)
print(f"Array 2D: {banda_vermelho.shape} -> 1D: {pixels_1d.shape}")

# Empilhando bandas — criando imagem multiespectral
# stack junta arrays existentes em uma nova dimensão
banda_azul = np.random.random((500, 500)) * 0.4 + 0.1
banda_verde = np.random.random((500, 500)) * 0.4 + 0.1
rgb = np.stack([banda_vermelho, banda_verde, banda_azul])
print(f"\nImagem RGB empilhada: {rgb.shape}")

# Transposição de eixos — (bandas, linhas, colunas) para (linhas, colunas, bandas)
rgb_t = np.transpose(rgb, (1, 2, 0))
print(f"RGB transposta: {rgb_t.shape}")

## 6. Cálculos Espaciais com Arrays

Usando NumPy para resolver problemas típicos de análise espacial.

In [ ]:
# Simulando pontos de coleta (estações de monitoramento ambiental)
n_pontos = 100
lat = np.random.uniform(-33.5, -30.5, n_pontos)
lon = np.random.uniform(-54.5, -50.5, n_pontos)
temperatura = np.random.normal(25, 5, n_pontos)  # temperatura em °C
precipitacao = np.random.exponential(5, n_pontos)  # precipitação em mm

print(f"Temperatura — média: {temperatura.mean():.1f}°C, std: {temperatura.std():.1f}")
print(f"Precipitação — média: {precipitacao.mean():.1f}mm, max: {precipitacao.max():.1f}")

# Distância euclidiana entre pontos (simplificada)
# Para coordenadas geográficas reais, usaríamos haversine
centro_lat, centro_lon = -32.0, -52.5
distancias = np.sqrt((lat - centro_lat)**2 + (lon - centro_lon)**2)
print(f"\nDistância média ao ponto central: {distancias.mean():.3f} graus")

# Filtrando estações próximas (< 0.5 graus do centro)
proximas = distancias < 0.5
print(f"Estações próximas (< 0.5°): {np.sum(proximas)}")
print(f"Temperatura média nas estações próximas: {temperatura[proximas].mean():.1f}°C")

## 7. Álgebra Linear para Geoprocessamento

Operações matriciais são fundamentais para transformações de coordenadas e processamento de imagens.

In [ ]:
# Normalização Min-Max (comum em pré-processamento de imagens)
def normalizar(array):
    return (array - array.min()) / (array.max() - array.min())

ndvi_norm = normalizar(ndvi)
print(f"NDVI normalizado — min: {ndvi_norm.min():.2f}, max: {ndvi_norm.max():.2f}")

# Produto escalar — correlação entre duas bandas
corr = np.corrcoef(banda_vermelho.reshape(-1), banda_nir.reshape(-1))[0, 1]
print(f"\nCorrelação Red-NIR: {corr:.4f}")

# Transformação de coordenadas simples (rotação 2D)
angulo = np.radians(30)  # 30 graus
rotacao = np.array([
    [np.cos(angulo), -np.sin(angulo)],
    [np.sin(angulo), np.cos(angulo)]
])
pontos = np.column_stack([lon[:5], lat[:5]])  # 5 pontos
transformados = pontos @ rotacao.T
print(f"\n5 pontos originais:\n{pontos}")
print(f"Pontos rotacionados:\n{transformados}")

## 8. Interpolação e Preenchimento de Dados

Dados de satélite frequentemente têm lacunas (nuvens, falhas do sensor). Vamos ver como preenchê-las.

In [ ]:
# Simulando dados com lacunas (valores NaN = sem dados)
dados_elevacao = np.random.normal(500, 200, (100, 100))

# Criando lacunas artificiais (simulando nuvens)
mascara_lacuna = np.random.random((100, 100)) < 0.15
dados_elevacao[mascara_lacuna] = np.nan

print(f"Total de pixels: {dados_elevacao.size}")
print(f"Pixels com lacunas: {np.sum(np.isnan(dados_elevacao))}")

# Preenchendo lacunas com a média dos vizinhos (técnica simples)
media_valores = np.nanmean(dados_elevacao)
dados_preenchidos = np.where(np.isnan(dados_elevacao), media_valores, dados_elevacao)
print(f"Média usada para preenchimento: {media_valores:.1f}")
print(f"Após preenchimento — NaN restantes: {np.sum(np.isnan(dados_preenchidos))}")

## Resumo

Neste módulo aprendemos:
- Criar arrays NumPy simulando imagens de satélite
- Realizar band math (NDVI e outros índices)
- Indexar e fatiar regiões de interesse
- Calcular estatísticas espaciais
- Manipular coordenadas
- Aplicar transformações matriciais
- Lidar com dados faltantes

O NumPy é a base sobre a qual bibliotecas como GDAL, Rasterio e xarray são construídas — todas essenciais para GeoAI.

# Desafio

## Monitoramento de Desmatamento com Índices Espectrais

Você é um analista de sensoriamento remoto no INPE (Instituto Nacional de Pesquisas Espaciais) e recebeu dados simulados de duas imagens de satélite da mesma região da Amazônia, com 6 meses de diferença.

**Tarefa:**

1. Crie dois arrays 200x200 representando o NDVI da região em duas datas diferentes (`ndvi_jan2024` e `ndvi_jul2024`). Use `np.random.seed(123)` para reprodutibilidade.
   - Janeiro deve ter valores mais altos de NDVI (estação chuvosa, vegetação exuberante)
   - Julho deve ter valores mais baixos (estação seca, possível desmatamento)

2. Calcule a diferença entre as duas imagens: `diferenca = ndvi_jan2024 - ndvi_jul2024`

3. Identifique as áreas de possível desmatamento onde a diferença de NDVI é maior que 0.3 (indicando perda significativa de vegetação)

4. Calcule:
   - A área total desmatada (número de pixels e percentual)
   - O NDVI médio na área desmatada em cada data
   - O valor máximo de NDVI perdido

5. Crie uma "máscara de alerta" — um array booleano marcando as áreas críticas

**Dica:** Use `np.random.uniform()` ou `np.random.normal()` para gerar os dados de NDVI com distribuições diferentes para cada data.